# BitLoop Stage C probe — can a free T4 reproduce the published baseline?

Runs the authors' own code for *Speed is Confidence* (Dillon 2026, Apache-2.0), experiment `x182m`: the fast Tiny Recursive Model baseline on Sudoku-Extreme (1,000 training puzzles, 8k steps at batch 384, reported 85.5% ± 1.3). This probe trains for at most 40 minutes to measure speed and memory, and to compare the early accuracy with the authors' published log. It decides whether a full reproduction fits a free session.

Third-party code: `https://github.com/rekursiv-ai/sic` pinned to commit `164f67765935`. Two changes only: the dataset builder is seeded (`20261120`) so the training subset is reproducible, and two short experiment files are added.

**Kaggle:** Accelerator **GPU T4 x2**, Internet **On**, then *Save Version → Save & Run All (Commit)*. About 1.5 hours. Download `stageC_probe.zip` from the Output tab afterwards.

In [ ]:
import torch, time, json, subprocess, os
n = torch.cuda.device_count(); assert n >= 1, 'No GPU: turn on the accelerator and rerun.'
print('torch', torch.__version__, '| GPUs:', [torch.cuda.get_device_name(i) for i in range(n)])
print('bf16 supported (reported):', torch.cuda.is_bf16_supported())
# Micro-benchmark: the published recipe keeps the whole model in bfloat16, which older GPUs lack natively
bench = {}
for name, dt in [('float32', torch.float32), ('float16', torch.float16), ('bfloat16', torch.bfloat16)]:
    try:
        a = torch.randn(384 * 81, 512, device='cuda', dtype=dt); w = torch.randn(512, 2048, device='cuda', dtype=dt)
        for i in range(13):
            if i == 3: torch.cuda.synchronize(); t = time.time()
            b = torch.nn.functional.silu(a @ w)
        torch.cuda.synchronize(); bench[name] = round((time.time() - t) / 10 * 1000, 2)
    except Exception as e:
        bench[name] = 'ERROR ' + repr(e)[:120]
print('ms per 31104x512 @ 512x2048 matmul + SiLU:', bench)
os.makedirs('probe_out', exist_ok=True); json.dump({'gpus': [torch.cuda.get_device_name(i) for i in range(n)], 'bench_ms': bench, 'torch': torch.__version__}, open('probe_out/gpu.json', 'w'))

In [ ]:
%%bash
set -e
curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
export PATH=$HOME/.local/bin:$PATH
rm -rf sic && git clone -q https://github.com/rekursiv-ai/sic sic && cd sic && git checkout -q 164f6776593516bb436aa8154799fcde1b97e163
test "$(git rev-parse HEAD)" = "164f6776593516bb436aa8154799fcde1b97e163" && echo "pinned commit OK: $(git log -1 --format='%h %ad' --date=short)"
head -3 LICENSE | tail -2
uv sync --no-dev --frozen 2>&1 | tail -3
.venv/bin/python -c "import sys, torch; print('venv python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
%%bash
set -e
cd sic
# Seed the builder so the 1,000-puzzle training subset and its augmentations are reproducible
sed -i 's/^rng = np.random.default_rng()$/rng = np.random.default_rng(20261120)/' code/tools/dataset/build_sudoku_dataset.py
grep -n 'default_rng(20261120)' code/tools/dataset/build_sudoku_dataset.py
export HF_HUB_DISABLE_PROGRESS_BARS=1
.venv/bin/python -m code.tools.dataset.build_sudoku_dataset --output-dir $PWD/../data/sudoku-extreme-1k-aug-1000 --subsample-size 1000 --num-aug 1000 2>&1 | tail -3
cd ../data/sudoku-extreme-1k-aug-1000 && find . -type f | sort | xargs sha256sum | tee ../../probe_out/dataset_sha256.txt | cut -c1-16,65- && du -sh .

In [ ]:
import os
data_dir = os.path.abspath('data/sudoku-extreme-1k-aug-1000')
open('sic/code/sudoku/zz_probe.py', 'w').write("\"\"\"BitLoop probe of x182m (authors' fast TRM baseline, seed 42): same schedule, stopped early by time.\"\"\"\nfrom experiment import main\nfrom x182m import Experiment as Base\n\n\nclass Experiment(Base):\n    data_dir: str = \"DATA_DIR\"\n    eval_every_steps: int = 100          # the training loop only prints timing at evals\n    max_eval_samples: int = 3_840        # authors use 38,400; kept small so the probe is cheap\n    max_eval_samples_train: int = 0\n    max_train_sec: float = 60 * 40       # 40 minutes of training time, evals excluded\n    checkpoint_steps: list[int] = []     # noqa: RUF012\n\n\nif __name__ == \"__main__\":\n    main(Experiment())\n".replace('DATA_DIR', data_dir))
open('sic/code/sudoku/zz_probe32.py', 'w').write("\"\"\"BitLoop probe: x182m with float32 parameters, to see what full precision costs on this GPU.\"\"\"\nfrom experiment import main\nfrom model import TRM3, TRM3ConfigProtocol\nfrom x182m import Experiment as Base\n\nimport torch\n\n\nclass Experiment(Base):\n    data_dir: str = \"DATA_DIR\"\n    eval_every_steps: int = 100\n    max_eval_samples: int = 3_840\n    max_eval_samples_train: int = 0\n    max_train_sec: float = 60 * 40\n    checkpoint_steps: list[int] = []     # noqa: RUF012\n\n    config: TRM3ConfigProtocol = TRM3.Config(\n        K_H=1,\n        K_L=1,\n        carry_H=\"all\",\n        carry_L=\"all\",\n        z_L_init_svd=True,\n        z_L_random_init=False,\n        dtype=torch.float32,\n    )\n\n\nif __name__ == \"__main__\":\n    main(Experiment())\n".replace('DATA_DIR', data_dir))
print('probe experiments written; data at', data_dir)

In [ ]:
%%bash
cd sic
nvidia-smi --query-gpu=index,memory.used --format=csv,noheader -l 20 > ../probe_out/mem.log 2>&1 &
MEM=$!
# GPU 0: the published recipe as is (bfloat16). GPU 1 (if present): float32 parameters.
CUDA_VISIBLE_DEVICES=0 .venv/bin/python code/sudoku/zz_probe.py > ../probe_out/run_bf16.stdout 2>&1 &
P0=$!
if [ "$(nvidia-smi -L | wc -l)" -ge 2 ]; then CUDA_VISIBLE_DEVICES=1 .venv/bin/python code/sudoku/zz_probe32.py > ../probe_out/run_fp32.stdout 2>&1 & P1=$!; fi
wait $P0; echo "bf16 run exit code: $?"
if [ -n "$P1" ]; then wait $P1; echo "fp32 run exit code: $?"; fi
kill $MEM 2>/dev/null
cp code/sudoku/zz_probe*.log ../probe_out/ 2>/dev/null; true

In [ ]:
import re, glob, json
ref = {100: None, 500: (60.72, 3.80), 1000: (67.05, 13.26)}   # authors' x182m log: cell %, puzzle % (38,400 test puzzles)
for tag in ['bf16', 'fp32']:
    try: txt = open(f'probe_out/run_{tag}.stdout', errors='replace').read()
    except FileNotFoundError: continue
    steps = re.findall(r'^Step\s*(\d+)\s+Test\s+([\d.]+)% /\s*([\d.]+)%.*?\(([\d.]+)s /\s*([\d.]+)s\)', txt, flags=re.M)
    print(f'=== {tag}: {len(steps)} evals')
    if not steps: print(txt[-1500:]); continue
    for s, cell, puz, ev, tr in steps[:3] + steps[-3:]:
        print(f'  step {int(s):5d}  cell {cell}%  puzzle {puz}%   eval {float(ev):6.1f}s   train-since-last-eval {float(tr):6.1f}s')
    tr = [float(x[4]) for x in steps[1:]]   # skip the first interval: it includes compilation
    if tr:
        sps = sum(tr) / (100 * len(tr)); print(f'  -> {sps:.2f} s/step after warm-up; 8,000 steps = {sps * 8000 / 3600:.1f} h of training')
        ev = [float(x[3]) for x in steps]; print(f'  -> eval of 3,840 puzzles: {sum(ev)/len(ev):.0f}s; a full 38,400-puzzle eval = {sum(ev)/len(ev)*10/60:.0f} min')
mem = {}
for line in open('probe_out/mem.log'):
    p = line.replace('MiB', '').split(',')
    if len(p) == 2 and p[1].strip().isdigit(): mem[p[0].strip()] = max(mem.get(p[0].strip(), 0), int(p[1]))
print('peak GPU memory (MiB) by GPU index:', mem)
print('authors (RTX 5090, bf16): 0.35 s/step, 304 s per 38,400-puzzle eval; step 500 = 60.72% cell / 3.80% puzzle')

In [ ]:
!zip -qr stageC_probe.zip probe_out && ls -la stageC_probe.zip